# 목표

연말정산 신고 안내 문서 활용 RAG 시스템 구현

In [301]:
# 기본 경로 설정
# ===============================================================
import os
import requests

PROJECT_NUM = "14"

ROOT_DIR = os.getcwd()


try:
    from google.colab import drive, userdata
    IS_COLAB_MODE = True
    print("코랩 모드")

except ModuleNotFoundError as e:
    IS_COLAB_MODE = False
    ROOT_DIR = os.path.abspath(os.path.join(ROOT_DIR, ".."))
    os.environ["KMP_DUPLICATE_LIB_OK"] = "True"
    os.environ["TOKENIZERS_PARALLELISM"] = "false"
    print(f"로컬 모드")


DATA_DIR = os.path.join(ROOT_DIR, "data")
RAW_DIR = os.path.join(DATA_DIR, "raw")
os.makedirs(DATA_DIR, exist_ok=True)


# 환경변수 로드 설정
def get_secret(key_name: str):
    if IS_COLAB_MODE:
        return userdata.get(key_name)
    else:
        from dotenv import load_dotenv
        load_dotenv(dotenv_path=os.path.join(ROOT_DIR, ".env"))
        return os.getenv(key_name)


if IS_COLAB_MODE:
    import subprocess

    drive.mount('/content/drive')

    # 압축 파일 확보
    if "raw.tar.gz" not in os.listdir():

        headers = {
            "Authorization": f"token {get_secret('GITHUB_PAT')}",
            "Accept": "application/vnd.github.v3+json"
        }

        release_url = f"https://api.github.com/repos/wonbywondev/ML-DL-data/releases/tags/data-v{PROJECT_NUM}"

        response = requests.get(release_url, headers=headers)
        asset_id = response.json().get('assets', [])[0]["id"]

        download_url = f"https://api.github.com/repos/wonbywondev/ML-DL-data/releases/assets/{asset_id}"
        download_headers = headers.copy()
        download_headers["Accept"] = "application/octet-stream"

        with requests.get(download_url, headers=download_headers, stream=True) as r:
            r.raise_for_status()
            with open("raw.tar.gz", 'wb') as f:
                for chunk in r.iter_content(chunk_size=8192):
                    f.write(chunk)

    print("· 압축 파일 있음")


    # 압축 해제 및 경로 처리
    if not os.path.exists(RAW_DIR):
        subprocess.run(["tar", "-xzvf", "raw.tar.gz", "-C", DATA_DIR], check=True)

    print("· 압축 해제 완료")
    print("· 환경 세팅 완료")


    DRIVE_DIR = os.path.join(ROOT_DIR, "drive", "MyDrive")
    SAVE_DIR = os.path.join(DRIVE_DIR, "runs", PROJECT_NUM)

    os.makedirs(SAVE_DIR, exist_ok=True)

PDF_PATH = os.path.join(RAW_DIR, "2024년+원천징수의무자를+위한+연말정산+신고안내.pdf")

로컬 모드


In [302]:
import re
import pandas as pd

import pdfplumber
from img2table.document import PDF
from img2table.ocr import TesseractOCR

from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.document_loaders import PyPDFLoader

### 텍스트 데이터

In [303]:
loader = PyPDFLoader(PDF_PATH)
docs = loader.load()

for text in docs:
    text.metadata = {"page": text.metadata["page"]}

In [334]:
# 문서 구조 기반 페이지 정리
DOCS_ABSTRACT = [docs[i] for i in range(2, 10)]
DOCS_INDEX = [docs[i] for i in range(10, 15)]
DOCS_REVISED = [docs[i] for i in range(16, 37)]
DOCS_CONCRETE = docs[39:]

### 요약 페이지(abstract)

In [305]:
# 메타데이터(chapter_title)
with pdfplumber.open(PDF_PATH) as pdf:
    for page in DOCS_ABSTRACT:
        page_num = page.metadata["page"] 

        head = pdf.pages[page_num].within_bbox((0, 0, 538, 130))
        tail = pdf.pages[page_num].within_bbox((0, 131, 538, 737))

        if head.extract_text():
            target_text = head.extract_text().replace("\n", " ")

        page.metadata["chapter_title"] = target_text
        page.page_content = tail.extract_text()

In [306]:
print(DOCS_ABSTRACT[0])

page_content='이용자 서비스 내 용 접근 경로
국세청 홈페이지(www.nts.go.kr) 
원천징수(연말정산)안내 홈페이지
국세신고안내  개인 또는 법인  연말정산
국세법령정보시스템
공통 연말정산 관련 질의회신 및 판례 조회
(www.hometax.go.kr  법령정보)
국세청 국세상담센터
인터넷 상담 및 회신
(www.hometax.go.kr  상담/제보)
홈택스  장려금·연말정산·기부금 
연말정산 소득·세액공제 자료 조회 연말정산간소화
[ 안내 ] 126-내선1-5번
소득공제
자료조회 현금영수증
(홈택스  전자(세금)계산서·현금영수증·
현금영수증 발행금액 조회
신용카드  현금영수증(근로자·소비자))
[ 안내 ] 126-내선1-1번
홈택스  장려금·연말정산·기부금 
공제신고서
간소화자료 선택 후 신고서 자동 반영 편리한 연말정산
작성
[ 안내 ] 126-내선1-5번
홈택스  장려금·연말정산·기부금 
작 성된 공제신고서 및 증명자료
간편제출 편리한 연말정산
온라인 제출
[ 안내 ] 126-내선1-5번
근로자
과거 원천징수 영수증(지급명세서)조회
* ’ 19년~’24년 및 ’25년 중 제출한
’25년 귀속분(중도퇴사자 등)은 조회 가능 홈택스  My홈택스  연말정산 
신고결과
* ’ 25.8월 수시오픈 이후부터는 지급명세서 등 제출내역
조회
’19년 귀속 확인 불가 [ 안내 ] 126-내선1-3번
제 출된 연말정산 신고사항은 제출
다음날부터 조회 가능
「근로자를 위한 연말정산 안내」 책자
안내책자 국세청 홈페이지(www.nts.go.kr) 
「근로자를 위한 연말정산」 동영상
및 영상 국세신고안내  개인 또는 법인  연말정산
「편리한 연말정산 이용방법」 동영상
국세청 홈택스(www.hometax.go.kr) 
프로그램 연말정산 모의계산 서비스 장려금·연말정산·기부금  편리한
연말정산  (모의계산) 연말정산 자동계산
원천징수이행상황신고 및 환급신청
원천세 전자신고 설명서
지급명세서 제출 홈택스(www.hometax.

### 목차 페이지(index)

In [307]:
# split 구분: 대제목
for page in DOCS_INDEX[:-1]:
    split = page.page_content.split("\n")
    index_list = [sentence for sentence in split if "·" in sentence]
    page.page_content = "\n".join(index_list)


chapter_splitter = RecursiveCharacterTextSplitter(
    chunk_size=300,
    chunk_overlap=0,
    separators=[r"\n(?=[가-힣])"],
    is_separator_regex=True,
    keep_separator=True
)

DOCS_INDEX = chapter_splitter.split_documents(DOCS_INDEX)

# 메타데이터(chapter_title)
for chapter in DOCS_INDEX:
    chapter.metadata["chapter_title"] = re.sub(
        r"[\s·.]{2,}\d+", "", 
        chapter.page_content.strip().split("\n")[0]
    )

In [308]:
print(DOCS_INDEX[0])

page_content='2024년 귀속 연말정산 개정세법 요약 ·······························1' metadata={'page': 10, 'chapter_title': '2024년 귀속 연말정산 개정세법 요약'}


### 개정안 페이지(revised)

In [309]:
# split 구분: 개정안 항목
chapter_splitter = RecursiveCharacterTextSplitter(
    chunk_size=100,
    chunk_overlap=0,
    separators=[r"\d+\s+.*?\n\(.*"],
    is_separator_regex=True,
    keep_separator=True
)

DOCS_REVISED = chapter_splitter.split_documents(DOCS_REVISED)

# 메타데이터(chapter_title)
del_list = list()
for chapter in DOCS_REVISED:
    if chapter.page_content == "원천징수의무자를 위한 \n2024년 연말정산 신고안내":
        del_list.append(chapter)
    elif chapter.page_content == "01. 2024년 귀속 연말정산 개정세법 요약":
        del_list.append(chapter)

    else:
        match = re.search(r"^\d+\s+(.*?)\n\(.*", chapter.page_content, re.DOTALL)
        if match:
            chapter_title = match.group(1).strip()
            chapter.metadata["chapter_title"] = chapter_title
            chapter.page_content = chapter.page_content.replace(chapter_title, "")

for del_ in del_list:
    DOCS_REVISED.remove(del_)

In [310]:
print(DOCS_REVISED[1])

page_content='1  
(소득세법 제12조 제3호 마목, 같은 법 시행령 제10조의2)
<개정취지> 육아휴직 지원
종          전 개          정
▢ 근로소득에서 비과세되는 육아휴직 급여·수당 ▢ 비과세 소득 확대
○ ｢고용보험법｣에 따라 받는 육아휴직급여 ○ (좌  동)
○ 공무원 또는 ｢사립학교교직원 연금법｣, ｢별정우체국법｣을
적용받는 사람이 관련 법령에 따라 받는 육아휴직수당
<추  가>    - 사립학교 직원이 사립학교 정관 등에 의해 지급받는 
월 150만원 이하의 육아휴직수당
<적용시기> 2024.1.1. 이후 지급받는 분부터 적용
' metadata={'page': 16, 'chapter_title': '육아휴직수당 비과세 적용대상 확대 및 범위 규정'}


In [311]:
for chap in DOCS_INDEX:
    print(chap.page_content)

2024년 귀속 연말정산 개정세법 요약 ·······························1

간소화서비스 전면 개편 ············································24
1. 주요 개선 내용 ···························································24
2. 소득금액 100만원(총급여 500만원) 산출 방법 ···········24
3. 주의사항 ·····································································24
2024년 귀속 연말정산 주요 일정 ·······························25
1. 회사의 연말정산 업무 일정 ·········································26
2. 원천징수의무자의 서류제출 의무 ································30
원천징수의무자의 연말정산 중점 확인사항 ·················34
1. 근로소득 원천징수 중점 확인사항(연말정산 이전) ·······34
2. 소득·세액공제 증명서류 중점 확인사항(연말정산 시) ··36
3. 연말정산 과다공제 주요 항목 ······································37
4. 잘못된 소득·세액공제에 따른 가산세 부담 ··················44

근로소득 ····································································48
1. 근로소득의 범위(소법 §20) ·········································48
2. 비과세 근로소득 등 ·····················································52
3. 일용근로소득과 일반근로소득의 구분 ·························73
4. 근로소득의 수입시기(소령 §49) ·····

### 상세 데이터(concrete)

In [347]:
chapter_title_map = {
    "2024년 귀속 연말정산 중점 추진사항": range(39, 62),
    "근로소득 연말정산": range(62, 224),
    "연말정산 종합사례 및 서식 작성방법": range(224, 348),
    "사업소득·연금소득 연말정산": range(348, 368),
    "종교인 소득 연말정산": range(368, 382),
    "연말정산 관련 서비스": range(382, 400),
    "연말정산 간소화 서비스": range(400, 409),
    "간소화자료 일괄제공 서비스": range(409, 414),
    "맞벌이부부 연말정산": range(414, 415),
    "연말정산 주요 용어 설명": range(415, 419),
    "소득·세액공제신고서 첨부서류": range(419, 426),
}

num_map = dict()
for key, value in chapter_title_map.items():
    for num in value:
        num_map[num] = key

In [348]:
for page in DOCS_CONCRETE:
    page.metadata["chapter_title"] = num_map[page.metadata["page"]]

In [349]:
DOCS_CONCRETE

[Document(metadata={'page': 39, 'chapter_title': '2024년 귀속 연말정산 중점 추진사항'}, page_content='24\nⅠ 간소화서비스 전면 개편 \n01 주요 개선 내용\n1. 간소화서비스를 개선하여 소득금액 100만원을(총급여 500만원) 초과①하는  비공제 대상 \n부양가족의 공제자료는 활용을 제한하기 위해 조회･다운로드 기능 미제공②\n     ① ’24년 6월까지 발생한 소득자료 기준\n     ② 보험료･의료비･교육비는 부양가족의 인적공제 가능 여부와 무관하게 모두 제공\n2 부양가족 중 소득기준 초과자(Y)를 간소화자료로 제공(Y만 표시, N은 미표시)\n소득초과 부양가족 정보 제공 간소화자료 (예시) \n2024년 귀속 소득기준 초과 부양가족 내역(소득 발생기간 : 2024년 1월~6월)\n관 계 성 명 주민등록번호 소득기준 초과\n배우자 이세정 820505-2****** Y\n부 김국세 551012-1****** Y\n     ③ 홈택스에서 기본공제자 체크 시 소득기준 초과여부를 한 번 더 확인하도록 안내하는 팝업 추가\n02 소득금액 100만원(총급여 500만원) 산출 방법\n1. 근로소득만 있는 경우에는 ’24.1월~’24.6월까지 발생한 총급여\n2. 타 소득이 있는 경우 아래의 방법으로 산정한 ’24.1월~’24.6월까지의 소득금액 합산\n※ 각 소득별 소득금액 산출방법\n    ① 근로소득 = 총급여액 - (총급여액 × 근로소득공제) ② 사업소득= 지급액- (지급액×업종별 단순경비율[타가])\n    ③ 기타소득 = 지급액 - (지급액 × 의제필요경비율) ④ 양도소득 = 양도소득금액(음수는 0으로 간주)\n    ⑤ 퇴직소득 = 과세대상 퇴직급여  \n03 주의사항\n1. ’24.1월~6월까지 발생한 근로소득, 사업소득, 기타소득, 퇴직소득, 양도소득(주식 제외) 이 있는 \n부양가족의 소득금액이 100만원(근로소득만 있는 경우 총급여 500만원)을 초과하는 경우 부양\n가족의 명단 제공\

### 표 데이터

In [313]:
# pdf = PDF(
#     PDF_PATH, 
#     detect_rotation=False,
#     pdf_text_extraction=True
# )

# ocr = TesseractOCR(n_threads=1, lang="eng")

# TABLES_BY_PAGE = pdf.extract_tables(
#     ocr=ocr,
#     implicit_rows=False,
#     implicit_columns=False,
#     borderless_tables=False,
#     min_confidence=40
# )

In [314]:
# # 표 데이터 - 메타데이터 title 보정

# TABLE_TITLE_LIST = list()
# for page_num, tables in TABLES_BY_PAGE.items():
#     for table in tables:
#         if table and table.title:
#             if len(table.title) > 25:
#                 table.title = None
#             else:
#                 TABLE_TITLE_LIST.append(table.title)

In [315]:
# # 표 데이터 metadata에 merge

# def trim_table(df: pd.DataFrame):
#     df.columns = df.iloc[0]
#     df = df[1:]
#     df.reset_index(drop=True, inplace=True)

#     return df


# to_delete_list = list()

# for page_num, tables in TABLES_BY_PAGE.items():
#     if tables:
#         docs[page_num].metadata["table"] = {
#             f"{page_num}.{i}": trim_table(table.df) for i, table in enumerate(tables)
#         }

#         for table in tables:
#             table = trim_table(table.df)

#     else:
#         docs[page_num].metadata["table"] = None
#         to_delete_list.append(page_num)

# for page_num in to_delete_list:
#     del TABLES_BY_PAGE[page_num]

pdf 육안 확인 + df로 바꿔보니 여간 복잡한 게 아니다.

1. 병합된 셀이 많다.
2. 특수문자(O) 같은 게 씹히는 경우가 있다.
3. 타이틀을 일일이 달아줘야 할 것 같다.
4. 표에서 확인할 수 있는 정보는 표를 참고하라고 따로 명령해야 할 듯.

In [316]:
"https://huggingface.co/microsoft/tapex-base-finetuned-wikisql"

'https://huggingface.co/microsoft/tapex-base-finetuned-wikisql'

## RAG 설계

0. 문장 데이터와 표 데이터로 나눈다.
1. 문장 데이터는 OPENAI 모델이 진행.
2. 표 데이터는 https://huggingface.co/microsoft/tapex-base-finetuned-wikisql
3. 허위 정보를 알려줘서는 안 되니, 모르는 건 모른다고 하자.
4. 참고한 페이지 정보를 같이 출력해주어 유저가 더블 체크할 수 있도록 하자.